In [14]:
from src.utils import load_config
from src.models.personalized_level_difficulty import MODEL_REGISTRY
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

config = load_config("configs/personalized_difficulty_level.yaml")


In [15]:
processed_data_path = config['processed_data_path']
if not os.path.exists(processed_data_path):
    print(f"❌ HATA: Gerekli bir dosya bulunamadı -> {processed_data_path}")

In [16]:
df = pd.read_excel(processed_data_path)

In [17]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
country,4656.0,1.386168,1.290939,0.0,0.0,1.000000,3.000000,3.00000
age,4656.0,39.458119,9.278665,18.0,35.0,39.000000,43.000000,99.00000
sex,4656.0,0.479381,0.499628,0.0,0.0,0.000000,1.000000,1.00000
total_spent_in_usd,4656.0,2.489220,7.023094,0.0,0.0,0.000000,0.000000,84.30002
total_spent_time,4656.0,0.562070,1.681009,0.0,0.0,0.000000,0.000000,23.00000
avg_daily_session,4656.0,2.856286,2.204622,1.0,1.5,2.181818,3.451136,28.00000
avg_weekly_session,4656.0,8.593687,10.782470,1.0,2.0,5.000000,10.750000,127.00000
easy_completion_rate,4656.0,93.943299,16.315900,50.0,100.0,100.000000,100.000000,100.00000
medium_completion_rate,4656.0,53.543814,38.080352,0.0,25.0,50.000000,100.000000,100.00000
hard_completion_rate,4656.0,5.530498,14.059026,0.0,0.0,0.000000,0.000000,100.00000


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4656 entries, 0 to 4655
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_key_id             4656 non-null   object 
 1   country                 4656 non-null   int64  
 2   age                     4656 non-null   int64  
 3   sex                     4656 non-null   int64  
 4   total_spent_in_usd      4656 non-null   float64
 5   total_spent_time        4656 non-null   int64  
 6   avg_daily_session       4656 non-null   float64
 7   avg_weekly_session      4656 non-null   float64
 8   easy_completion_rate    4656 non-null   int64  
 9   medium_completion_rate  4656 non-null   int64  
 10  hard_completion_rate    4656 non-null   float64
dtypes: float64(4), int64(6), object(1)
memory usage: 400.3+ KB


In [19]:
import mlflow
import mlflow.sklearn
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

output_dir = "./mlruns/level_difficulty"
os.makedirs(output_dir, exist_ok=True)

label_encoders = {}
for col in ['country', 'sex']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    label_encoders[col] = list(le.classes_)

# Scale features once
features = [
    'country', 'age', 'sex',
    'total_spent_in_usd', 'total_spent_time',
    'avg_daily_session', 'avg_weekly_session',
    'easy_completion_rate', 'medium_completion_rate', 'hard_completion_rate'
]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(df[features])

pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(features_scaled)

mlflow.set_experiment(config['experiment_name'])

for model_config in config['models']:
    model_name = model_config['model_name']
    ModelClass = MODEL_REGISTRY[model_name]

    for params in model_config['param_grid']:
        with mlflow.start_run(run_name=f"{model_name}"):


            mlflow.log_param("model_name", model_name)
            mlflow.log_params(params)
        

            model = ModelClass(params)

            df['cluster'] = model.fit_predict(X_pca)

            cluster_label_map = {
                0: 'Easy',
                1: 'Normal',
                2: 'Hard',
            }
            df['user_level_difficulty_category'] = df['cluster'].map(cluster_label_map)

            # Save result to CSV
            result_path = os.path.join(output_dir, f"{model_name}_{str(params)}_results.csv".replace(" ", "_").replace(":", ""))
            df[['user_key_id', 'cluster', 'user_level_difficulty_category']].to_csv(result_path, index=False)
            mlflow.log_artifact(result_path)

            labels = df['cluster']
            num_clusters = labels.nunique()

            if num_clusters >= 2:
                try:
                    sil = silhouette_score(X_pca, labels)
                    ch = calinski_harabasz_score(X_pca, labels)
                    db = davies_bouldin_score(X_pca, labels)


                    mlflow.log_metric("silhouette_score", sil)
                    mlflow.log_metric("calinski_harabasz_score", ch)
                    mlflow.log_metric("davies_bouldin_score", db)
                except:
                    print(f"Metric logging failed: {e}")
                    

                cluster_counts = df['cluster'].value_counts().to_dict()
                for cluster_id, count in cluster_counts.items():
                    mlflow.log_metric(f"cluster_{cluster_id}_size", count)

            try:
                mlflow.sklearn.log_model(model.model, name="model")
            
                mlflow.log_dict(label_encoders, "level_difficulty_label_encoders.json")

                run_info = {
                    "training_columns": list(features)
                }
                mlflow.log_dict(run_info, "level_difficulty_run_info.json")
            except Exception as e:
                print(f"Model logging failed: {e}")

            print(f"Logged {model_name} with params {params}")


Traceback (most recent call last):
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 347, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 445, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 1588, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 1581, in _read_helper
    result = read_yaml(root, file_name)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3

Logged KMeansClusterer with params {'n_clusters': 2, 'init': 'k-means++', 'n_init': 10, 'max_iter': 300, 'tol': 0.0001, 'algorithm': 'lloyd'}


2025/06/21 21:40:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged KMeansClusterer with params {'n_clusters': 3, 'init': 'k-means++', 'n_init': 10, 'max_iter': 300, 'tol': 0.0001, 'algorithm': 'lloyd'}


2025/06/21 21:40:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

Logged KMeansClusterer with params {'n_clusters': 7, 'init': 'k-means++', 'n_init': 20, 'max_iter': 1000, 'tol': 0.001, 'algorithm': 'elkan'}


2025/06/21 21:40:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged KMeansClusterer with params {'n_clusters': 6, 'init': 'random', 'n_init': 10, 'max_iter': 300, 'tol': 0.0001, 'algorithm': 'lloyd'}


2025/06/21 21:40:47 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/06/21 21:40:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged SpectralClusterer with params {'n_clusters': 7, 'affinity': 'nearest_neighbors', 'assign_labels': 'kmeans', 'random_state': 42}
